# Lab 02: Inverses, Linear Systems & Span

**Reference:** Goodfellow et al. *Deep Learning*, Chapter 2, Sections 2.3–2.4

---

## What this lab covers

In this lab we build working intuition for two tightly connected ideas from the Deep Learning textbook:

| Section | Core question | Key tools |
|---------|---------------|-----------|
| **2.3** – Identity & Inverse Matrices | *Given $Ax = b$, when and how can we find $x$?* | Inverse, `np.linalg.solve`, condition number |
| **2.4** – Linear Dependence & Span | *Which right-hand sides $b$ even have a solution?* | Span, rank, linear independence |

Every concept here appears directly when we train ML models:
- Solving the **normal equations** for linear regression requires inverting $X^\top X$.
- The **condition number** tells us whether our features are too correlated for a stable solution.
- Neural networks are **massively underdetermined** systems – understanding the space of solutions is fundamental to understanding why deep learning generalises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import time
import warnings

np.random.seed(42)
print('Setup complete.')

---
## Part 1: Identity and Inverse Matrices (Section 2.3)

### The identity matrix: the "do nothing" transformation

The **identity matrix** $I_n$ is the $n \times n$ matrix with ones on the diagonal and zeros elsewhere:

$$I_n = \begin{pmatrix} 1 & 0 & \cdots & 0 \\\\ 0 & 1 & \cdots & 0 \\\\ \vdots & & \ddots & \vdots \\\\ 0 & 0 & \cdots & 1 \end{pmatrix}$$

Its defining property is that it leaves every vector unchanged:

$$I_n \, x = x \quad \forall \, x \in \mathbb{R}^n$$

Think of $I_n$ as the matrix analogue of multiplying by $1$ in scalar arithmetic.

### The matrix inverse

The **inverse** of a square matrix $A$ is the unique matrix $A^{-1}$ satisfying:

$$A^{-1} A = I_n \qquad \text{and} \qquad A A^{-1} = I_n$$

If such a matrix exists, we call $A$ **invertible** (or **non-singular**).

**Why do we care?** Because if $A$ is invertible, we can solve any linear system $Ax = b$ in one shot:

$$Ax = b \;\;\Longrightarrow\;\; x = A^{-1} b$$

> **Crucial caveat:** Not every matrix has an inverse. Only **square, full-rank** matrices are invertible. We will explore exactly when inversion fails later in this lab.

### Why `np.linalg.solve` beats `np.linalg.inv`

In theory, computing $x = A^{-1}b$ is clean and elegant. In practice, it is a **bad idea** for two reasons:

1. **Speed.** Computing $A^{-1}$ requires $O(n^3)$ operations, and then multiplying $A^{-1}b$ is another $O(n^2)$. The function `np.linalg.solve` uses **LU decomposition** to go directly from $A$ and $b$ to $x$, skipping the full inverse.

2. **Numerical stability.** Computing $A^{-1}$ explicitly can **square the condition number** of the problem. This means floating-point rounding errors are amplified much more than they need to be.

The next cell demonstrates both effects on a concrete 500×500 system.

In [ ]:
# Demo: np.linalg.solve vs np.linalg.inv for solving Ax = b
# Goodfellow 2.3: In practice, solve() is preferred over explicitly computing A^{-1}.

np.random.seed(42)
n = 500
A_demo = np.random.randn(n, n)  # random square matrix (almost certainly invertible)
b_demo = np.random.randn(n)     # random right-hand side

# Method 1: np.linalg.solve  --  uses LU factorisation internally
# This factors A = PLU once, then solves by forward/back substitution.
runs = 10
t0 = time.perf_counter()
for _ in range(runs):
    x_solve = np.linalg.solve(A_demo, b_demo)
elapsed_solve = (time.perf_counter() - t0) / runs * 1e3

# Method 2: Explicitly compute A^{-1}, then multiply
# This does MORE work: first compute the full n x n inverse, then matrix-vector multiply.
t0 = time.perf_counter()
for _ in range(runs):
    A_inv = np.linalg.inv(A_demo)
    x_inv = A_inv @ b_demo
elapsed_inv = (time.perf_counter() - t0) / runs * 1e3

# Compare: both should give the same answer, but solve() is faster
print(f'Matrix size : {n} x {n}')
print(f'np.linalg.solve : {elapsed_solve:.2f} ms  (residual ||Ax-b||={np.linalg.norm(A_demo @ x_solve - b_demo):.2e})')
print(f'np.linalg.inv   : {elapsed_inv:.2f} ms  (residual ||Ax-b||={np.linalg.norm(A_demo @ x_inv  - b_demo):.2e})')
print(f'Speed-up        : {elapsed_inv / elapsed_solve:.1f}x')
print()
print('Key insight (Book 2.3): solve() uses LU decomposition and avoids the extra')
print('matrix-matrix multiply, so it is faster and numerically more stable.')

### What we observed

- `np.linalg.solve` is roughly **1.5x faster** because it avoids computing the full $n \times n$ inverse matrix.
- Both methods produce residuals near machine epsilon ($\approx 10^{-11}$), but for ill-conditioned matrices the gap would be much larger.
- **Rule of thumb:** Never compute $A^{-1}$ just to multiply by $b$. Use `solve()` instead.

---
### Exercise 1: 2×2 Inverse by Formula

For a $2 \times 2$ matrix, there is a beautiful closed-form inverse:

$$A = \begin{pmatrix} a & b \\\\ c & d \end{pmatrix}
\qquad \Longrightarrow \qquad
A^{-1} = \frac{1}{ad - bc}\begin{pmatrix} d & -b \\\\ -c & a \end{pmatrix}$$

The scalar $\det(A) = ad - bc$ is the **determinant**. Geometrically, $|\det(A)|$ is the area of the parallelogram formed by the columns of $A$. When $\det(A) = 0$, the columns are collinear (linearly dependent), the parallelogram collapses to a line, and $A$ has no inverse.

**Intuition:** The inverse formula swaps the diagonal elements, negates the off-diagonal elements, and divides by the determinant. This "undoes" what $A$ did to space.

Implement this formula and verify it against `np.linalg.inv` on three test matrices.

In [ ]:
def inverse_2x2(A):
    """
    Return the inverse of a 2x2 matrix using the closed-form formula.
    
    Parameters
    ----------
    A : np.ndarray, shape (2, 2)
    
    Returns
    -------
    np.ndarray, shape (2, 2)  -- the inverse of A
    
    Raises
    ------
    ValueError if A is singular (det == 0).
    """
    a, b = A[0, 0], A[0, 1]
    c, d = A[1, 0], A[1, 1]
    det = a * d - b * c           # determinant: area scaling factor
    if abs(det) < 1e-12:
        raise ValueError("Matrix is singular")
    # Swap diagonal, negate off-diagonal, divide by det
    return np.array([[d, -b], [-c, a]]) / det


# --- Verification ---
test_matrices = [
    np.array([[2.0, 1.0], [5.0, 3.0]]),
    np.array([[4.0, 7.0], [2.0, 6.0]]),
    np.array([[-3.0,  2.0], [ 1.0, -1.0]]),
]

all_passed = True
for i, M in enumerate(test_matrices):
    A_inv_formula = inverse_2x2(M)
    A_inv_numpy   = np.linalg.inv(M)
    match = np.allclose(A_inv_formula, A_inv_numpy, atol=1e-10)
    print(f'Test matrix {i+1}: match={match}, max_diff={np.max(np.abs(A_inv_formula - A_inv_numpy)):.2e}')
    all_passed = all_passed and match

print()
if all_passed:
    print('All tests passed.')
else:
    print('Some tests failed -- revisit your implementation.')

### What we observed

Our hand-rolled formula matches NumPy's inverse to within $\sim 10^{-15}$ (machine epsilon). The tiny discrepancies are from floating-point arithmetic – they are not mathematical errors.

> **Note:** This closed-form formula only works for $2 \times 2$ matrices. For larger matrices, there is no practical closed form – we need algorithmic methods like Gauss-Jordan elimination (next exercise) or LU decomposition.

---
### Exercise 2: Gauss-Jordan Elimination

The general algorithm for computing $A^{-1}$ is **Gauss-Jordan elimination**. The idea is wonderfully simple:

1. **Form the augmented matrix** $[A \mid I_n]$.
2. **Apply row operations** to transform the left half into $I_n$.
3. Whatever operations turned $A$ into $I_n$ will simultaneously turn $I_n$ into $A^{-1}$.

$$[A \mid I_n] \;\xrightarrow{\text{row operations}}\; [I_n \mid A^{-1}]$$

**Why does this work?** Each row operation is equivalent to left-multiplying by an elementary matrix $E_i$. After all operations:
$$E_k \cdots E_2 E_1 A = I_n \quad \Longrightarrow \quad E_k \cdots E_2 E_1 = A^{-1}$$
And the right half records exactly $E_k \cdots E_2 E_1 \cdot I_n = A^{-1}$.

**Partial pivoting** (swapping rows to put the largest element on the diagonal) is critical for numerical stability – it prevents dividing by tiny numbers that amplify rounding errors.

Implement this and test on $3 \times 3$ and $4 \times 4$ matrices.

In [ ]:
def gauss_jordan_inverse(A):
    """
    Compute A^{-1} via Gauss-Jordan elimination on [A | I].
    
    Parameters
    ----------
    A : np.ndarray, shape (n, n)
    
    Returns
    -------
    np.ndarray, shape (n, n)  -- the inverse of A
    
    Raises
    ------
    ValueError if A is singular.
    """
    n = A.shape[0]
    # Step 1: Augmented matrix [A | I]
    aug = np.hstack([A.astype(float), np.eye(n)])

    for col in range(n):
        # Partial pivoting: swap row to put largest |value| on the diagonal
        max_row = col + np.argmax(np.abs(aug[col:, col]))
        aug[[col, max_row]] = aug[[max_row, col]]

        if abs(aug[col, col]) < 1e-12:
            raise ValueError("Matrix is singular")

        # Step 2: Scale pivot row so pivot element becomes 1
        aug[col] = aug[col] / aug[col, col]

        # Step 3: Eliminate all other rows in this column
        for row in range(n):
            if row != col:
                aug[row] -= aug[row, col] * aug[col]

    # Step 4: Right half is now A^{-1}
    return aug[:, n:]


# --- Verification ---
np.random.seed(42)
test_cases = [
    np.array([[2.0, 1.0, 0.0],
              [1.0, 3.0, 1.0],
              [0.0, 1.0, 2.0]]),
    np.random.randn(4, 4),
]

all_passed = True
for i, M in enumerate(test_cases):
    A_inv_gj    = gauss_jordan_inverse(M)
    A_inv_numpy = np.linalg.inv(M)
    match = np.allclose(A_inv_gj, A_inv_numpy, atol=1e-10)
    print(f'Test {i+1} ({M.shape[0]}x{M.shape[0]}): match={match}, max_diff={np.max(np.abs(A_inv_gj - A_inv_numpy)):.2e}')
    all_passed = all_passed and match

print()
print('All tests passed.' if all_passed else 'Some tests failed.')

### What we observed

Our hand-written Gauss-Jordan elimination produces results matching NumPy's built-in `inv()` to machine precision. The algorithm has $O(n^3)$ complexity – the same as LU decomposition, but LU is preferred in practice because it is more numerically stable and can be reused for multiple right-hand sides.

> **ML context:** You would never implement Gauss-Jordan in a production ML system, but understanding row reduction helps you reason about rank, solvability, and what happens when systems break down.

---
### Exercise 3: When Does the Inverse Fail?

A matrix is **singular** (non-invertible) when its rows (or columns) are **linearly dependent**. This means at least one row can be written as a combination of the others, so the matrix "collapses" a dimension.

There are three equivalent ways to detect singularity:

| Test | Singular if... | Intuition |
|------|----------------|----------|
| Determinant | $\det(A) = 0$ | The volume scaling factor is zero – space is flattened |
| Rank | $\text{rank}(A) < n$ | Some columns are redundant |
| Condition number | $\kappa(A) = \infty$ | Inversion is infinitely sensitive to perturbations |

Let's build a singular matrix and watch all three diagnostics fire.

In [ ]:
# Step 1 -- build a singular matrix (row 1 = 2 * row 0)
A_singular = np.array([[1.0, 2.0, 3.0],
                        [2.0, 4.0, 6.0],   # row 1 = 2 * row 0  (linearly dependent!)
                        [0.0, 1.0, 1.0]])

# Step 2 -- determinant and condition number
det_val  = np.linalg.det(A_singular)
cond_val = np.linalg.cond(A_singular)

# Step 3 -- try to invert; numpy may not raise but the result is garbage
try:
    inv = np.linalg.inv(A_singular)
    print("Warning: numpy returned an inverse, but it's garbage:")
    print(f"  A @ A_inv = \n{A_singular @ inv}")
    print(f"  (Should be identity but isn't -- check the large values)")
except np.linalg.LinAlgError as e:
    print(f"np.linalg.inv raised: {e}")

# Step 4 -- interpretation
print(f"\ndet(A)  = {det_val:.2e}   (approx 0 means singular)")
print(f"cond(A) = {cond_val:.2e}  (huge means ill-conditioned / singular)")
print()
print("Interpretation:")
print("  - det approx 0 confirms the matrix is (numerically) singular.")
print("  - A condition number of inf (or very large) means a tiny change in b")
print("    can produce an arbitrarily large change in x -- the system is unstable.")
print("  - Root cause: row 1 is linearly dependent on row 0, so rank(A) = 2 < 3.")

### What we observed

NumPy does not always raise an error for singular matrices – it may return a "garbage" inverse filled with enormous numbers. The product $A \cdot A^{-1}$ is far from the identity. This is why checking the **determinant** and **condition number** before trusting an inverse is essential.

> **Key takeaway:** A matrix being "close to singular" (large $\kappa$) is often more dangerous than being exactly singular, because you get an answer that *looks* reasonable but is actually dominated by numerical noise.

---
### The condition number: quantifying numerical danger

The **condition number** $\kappa(A)$ measures how sensitive the solution $x$ of $Ax = b$ is to small perturbations in $b$:

$$\frac{\|\delta x\|}{\|x\|} \leq \kappa(A) \cdot \frac{\|\delta b\|}{\|b\|}$$

In terms of the singular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_n$:

$$\kappa(A) = \frac{\sigma_{\max}}{\sigma_{\min}}$$

| $\kappa(A)$ | Interpretation |
|-------------|----------------|
| $\approx 1$ | Well-conditioned. Errors are not amplified. |
| $10^4$ – $10^8$ | Moderately ill-conditioned. You lose $\log_{10}(\kappa)$ digits of precision. |
| $> 1/\epsilon_{\text{machine}}$ | Effectively singular. The solution is pure noise. |

**ML Connection:** In linear regression, the normal equations involve $(X^\top X)^{-1}$. When features are highly correlated, $X^\top X$ becomes ill-conditioned ($\kappa$ is huge). This is exactly why **regularization** (e.g., Ridge regression: $(X^\top X + \lambda I)^{-1}$) helps – adding $\lambda I$ to the diagonal raises $\sigma_{\min}$, reducing $\kappa$.

The following exercise constructs matrices with controlled condition numbers and measures how the solution error grows.

### Exercise 7: Condition Number Experiment

We will:
1. Construct matrices with known condition numbers $\kappa$ from $1$ to $10^{12}$.
2. Solve $Ax = b$, then solve $A(x + \delta x) = b + \delta b$ with a tiny perturbation.
3. Measure the relative error and compare it to the theoretical bound $\kappa \cdot \|\delta b\|/\|b\|$.

In [ ]:
np.random.seed(42)
n = 20
perturbation_size = 1e-4

def make_matrix_with_condition_number(n, kappa, rng):
    """
    Construct an n x n matrix with a specified condition number.

    Strategy: A = U @ diag(s) @ V.T where U, V are random orthogonal matrices
    and s is a vector of singular values with s[0]/s[-1] = kappa.
    This gives us precise control over kappa.
    """
    U, _ = np.linalg.qr(rng.standard_normal((n, n)))  # random orthogonal U
    V, _ = np.linalg.qr(rng.standard_normal((n, n)))  # random orthogonal V
    # Geometric spacing: largest singular value = kappa, smallest = 1
    s = np.geomspace(kappa, 1, n)  # sigma_max = kappa, sigma_min = 1
    return U @ np.diag(s) @ V.T   # kappa(A) = kappa by construction


# --- Experiment: sweep kappa from 1 to 10^12 ---
condition_numbers = np.logspace(0, 12, 25)
rng = np.random.default_rng(42)

rel_errors    = []
theory_bounds = []

b_true = np.ones(n)
# Fixed perturbation direction, scaled to desired relative size
delta_b = rng.standard_normal(n)
delta_b = delta_b / np.linalg.norm(delta_b) * perturbation_size * np.linalg.norm(b_true)

for kappa_target in condition_numbers:
    A = make_matrix_with_condition_number(n, kappa_target, rng)

    # Solve the original and perturbed systems
    x = np.linalg.solve(A, b_true)
    x_perturbed = np.linalg.solve(A, b_true + delta_b)

    # Measure relative error: ||delta_x|| / ||x||
    rel_err = (np.linalg.norm(x_perturbed - x) / np.linalg.norm(x))
    rel_errors.append(rel_err)

    # Theoretical upper bound: kappa * ||delta_b|| / ||b||
    kappa_actual = np.linalg.cond(A)
    bound = kappa_actual * (np.linalg.norm(delta_b) / np.linalg.norm(b_true))
    theory_bounds.append(bound)

rel_errors    = np.array(rel_errors)
theory_bounds = np.array(theory_bounds)

# --- Plot ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(condition_numbers, rel_errors,    'o-', color='royalblue',  lw=1.5, label='Measured relative error')
ax.loglog(condition_numbers, theory_bounds, 's--', color='darkorange', lw=1.5, label=r'Theory bound $\kappa \cdot (\|\delta b\|/\|b\|)$')
ax.axhline(perturbation_size, color='grey', lw=1, ls=':', label=f'Perturbation size {perturbation_size}')
ax.axvline(1/np.finfo(float).eps, color='red', lw=1, ls=':', label='Machine epsilon threshold')
ax.set_xlabel(r'Condition number $\kappa(A)$', fontsize=12)
ax.set_ylabel('Relative solution error', fontsize=12)
ax.set_title('Condition Number vs Solution Perturbation (Book 2.3)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Machine epsilon  : {np.finfo(float).eps:.2e}')
print(f'At kappa ~ 1e8, relative error ~ {rel_errors[np.argmin(np.abs(condition_numbers - 1e8))]:.2e}')
print('Beyond kappa ~ 1/eps the solution becomes meaningless (numerical noise dominates).')

### What we observed

The plot confirms the textbook result: the relative error in $x$ grows **linearly** with the condition number $\kappa(A)$. Once $\kappa$ approaches $1/\epsilon_{\text{machine}} \approx 10^{16}$, the solution is essentially random noise.

**Practical rule:** If your condition number is $10^k$, you lose roughly $k$ decimal digits of precision in the solution. With double-precision floats ($\sim 16$ digits), $\kappa = 10^{8}$ means only 8 reliable digits remain.

> **ML Connection revisited:** When you compute the normal equations $(X^\top X)^{-1} X^\top y$ and your features are highly correlated (multicollinear), $\kappa(X^\top X)$ can easily reach $10^{10}$ or higher. Ridge regression adds $\lambda I$ to the diagonal, which raises all singular values by $\lambda$. This caps $\kappa$ at roughly $\sigma_{\max}^2 / \lambda$, making the system well-conditioned.

---
## Part 2: Linear Dependence and Span (Section 2.4)

Now we shift from *"can we solve $Ax = b$?"* to the deeper question: *"which $b$ have solutions?"*

### Linear combination

A **linear combination** of vectors $v_1, v_2, \ldots, v_k$ is any expression of the form:

$$\sum_{i=1}^{k} c_i \, v_i = c_1 v_1 + c_2 v_2 + \cdots + c_k v_k$$

where $c_i \in \mathbb{R}$ are scalar **coefficients**. This is the most fundamental operation in linear algebra: scaling vectors and adding them together.

### Span

The **span** of a set of vectors is the set of *all* points reachable by their linear combinations:

$$\text{span}(\{v_1, \ldots, v_k\}) = \left\{ \sum_{i=1}^{k} c_i v_i \;\middle|\; c_i \in \mathbb{R} \right\}$$

Geometrically:
- The span of **one** non-zero vector is a **line** through the origin.
- The span of **two** non-collinear vectors is a **plane** through the origin.
- The span of **$n$** linearly independent vectors in $\mathbb{R}^n$ is **all of $\mathbb{R}^n$**.

### The column space connection

Here is the critical link to solving $Ax = b$. When we write $Ax$, we are computing a linear combination of the **columns** of $A$:

$$Ax = x_1 \underbrace{a_1}_{\text{col 1}} + x_2 \underbrace{a_2}_{\text{col 2}} + \cdots + x_n \underbrace{a_n}_{\text{col } n}$$

Therefore:

> **$Ax = b$ has a solution if and only if $b$ lies in the column space (span of columns) of $A$.**

This is the geometric essence of solvability.

### Visualising span in 2D

Let's see two cases:
- **Two non-collinear vectors:** Their span fills all of $\mathbb{R}^2$. Any 2D target $b$ can be reached.
- **Two collinear vectors:** Their span is just a line. Most targets $b$ cannot be reached.

In [ ]:
# Demo: Visualising span in 2-D
# Case 1 -- two non-collinear vectors span all of R^2.
# Case 2 -- two collinear vectors span only a line through the origin.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ------------------------------------------------------------------ #
# Case 1: full span (R^2)
# ------------------------------------------------------------------ #
ax = axes[0]
v1 = np.array([2.0, 0.5])   # first basis direction
v2 = np.array([0.5, 2.0])   # second basis direction (NOT parallel to v1)

# Shade the span -- since it fills all of R^2, we draw a large parallelogram
scale = 3.5
corners = np.array([
    -scale*v1 - scale*v2,
     scale*v1 - scale*v2,
     scale*v1 + scale*v2,
    -scale*v1 + scale*v2,
])
span_poly = plt.Polygon(corners, color='lightblue', alpha=0.4, zorder=0)
ax.add_patch(span_poly)

ax.quiver(0, 0, v1[0], v1[1], angles='xy', scale_units='xy', scale=1,
          color='royalblue', width=0.015, label=r'$v_1 = (2, 0.5)$')
ax.quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1,
          color='darkorange', width=0.015, label=r'$v_2 = (0.5, 2)$')
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.axhline(0, color='grey', lw=0.8)
ax.axvline(0, color='grey', lw=0.8)
ax.set_title('Case 1: Two non-collinear vectors\nSpan = all of $\\mathbb{R}^2$', fontsize=12)
ax.legend(loc='upper left')
ax.set_aspect('equal')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

# ------------------------------------------------------------------ #
# Case 2: collinear -- span is a single line
# ------------------------------------------------------------------ #
ax = axes[1]
u1 = np.array([1.5, 1.0])
u2 = np.array([3.0, 2.0])   # u2 = 2 * u1  (linearly dependent!)

# Draw the spanned line (thin band for visibility)
t = np.linspace(-2.2, 2.2, 200)
line_pts = np.outer(t, u1)
ax.fill_between(line_pts[:, 0], line_pts[:, 1] - 0.06,
                line_pts[:, 1] + 0.06, color='lightblue', alpha=0.5,
                label='Span (a line)', zorder=0)

ax.quiver(0, 0, u1[0], u1[1], angles='xy', scale_units='xy', scale=1,
          color='royalblue', width=0.015, label=r'$u_1 = (1.5, 1)$')
ax.quiver(0, 0, u2[0], u2[1], angles='xy', scale_units='xy', scale=1,
          color='darkorange', width=0.015, label=r'$u_2 = (3, 2) = 2u_1$')
ax.set_xlim(-3.5, 4.5)
ax.set_ylim(-3.0, 3.5)
ax.axhline(0, color='grey', lw=0.8)
ax.axvline(0, color='grey', lw=0.8)
ax.set_title('Case 2: Two collinear vectors\nSpan = a line (rank 1)', fontsize=12)
ax.legend(loc='upper left')
ax.set_aspect('equal')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

plt.suptitle('Visualising Span (Book 2.4)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### What we observed

- **Left panel:** Two non-collinear vectors span all of $\mathbb{R}^2$. Any point in the plane can be reached by some linear combination $c_1 v_1 + c_2 v_2$. The system $Ax = b$ has a unique solution for every $b$.

- **Right panel:** Two collinear vectors ($u_2 = 2u_1$) span only a 1D line. Most points in $\mathbb{R}^2$ are unreachable. Adding a second vector that is just a scaled copy of the first adds **no new directions** – it is redundant. This is the geometric meaning of linear dependence.

---
### Linear dependence and independence

A set of vectors $\{v_1, \ldots, v_k\}$ is **linearly dependent** if at least one of them can be written as a linear combination of the others. Equivalently, there exist scalars $c_1, \ldots, c_k$ (not all zero) such that:

$$c_1 v_1 + c_2 v_2 + \cdots + c_k v_k = 0$$

If the *only* solution to this equation is $c_1 = c_2 = \cdots = c_k = 0$, the vectors are **linearly independent**.

**Geometric intuition:**
- Independent vectors point in "genuinely different" directions.
- Dependent vectors are redundant – at least one is a combination of the others.

### Rank

The **rank** of a matrix $A$ is the number of linearly independent columns (which always equals the number of linearly independent rows):

$$\text{rank}(A) = \text{rank}(A^\top)$$

Rank tells us the dimensionality of the column space – how many "independent directions" the transformation $A$ covers.

**Quick test:** A set of $k$ vectors in $\mathbb{R}^n$ is linearly independent if and only if the matrix formed by stacking them as columns has rank $k$.

### Exercise 4: Check Linear Independence

Implement `is_linearly_independent(vectors)` using the rank test and verify on these cases:
- 2D: $\{(1,0), (0,1)\}$ – independent (standard basis)
- 2D: $\{(1,2), (2,4)\}$ – dependent (second = 2 times first)
- 3D: $\{(1,0,0), (0,1,0), (0,0,1)\}$ – independent (standard basis)
- 3D: $\{(1,2,3), (4,5,6), (5,7,9)\}$ – dependent (third = sum of first two)

In [ ]:
def is_linearly_independent(vectors, tol=1e-10):
    """
    Determine whether a list of vectors is linearly independent.

    Parameters
    ----------
    vectors : list of array-like, each of shape (n,)
    tol     : tolerance for rank computation

    Returns
    -------
    bool  -- True if the vectors are linearly independent.
    """
    # Stack vectors as columns; rank == number of vectors iff independent
    M = np.column_stack(vectors)
    return np.linalg.matrix_rank(M, tol=tol) == len(vectors)


# --- Verification ---
test_cases = [
    (np.array([[1, 0], [0, 1]]),          True,  '2D standard basis'),
    (np.array([[1, 2], [2, 4]]),           False, '2D collinear (dependent)'),
    (np.array([[1,0,0],[0,1,0],[0,0,1]]), True,  '3D standard basis'),
    (np.array([[1,2,3],[4,5,6],[5,7,9]]), False, '3D row3 = row1+row2 (dependent)'),
]

all_passed = True
for vecs, expected, label in test_cases:
    result = is_linearly_independent(vecs)
    status = 'PASS' if result == expected else 'FAIL'
    print(f'[{status}] {label}: got {result}, expected {expected}')
    all_passed = all_passed and (result == expected)

print()
print('All tests passed.' if all_passed else 'Some tests failed.')

### What we observed

The rank test correctly identifies all four cases. Notice that in the 3D dependent case, the third vector $(5,7,9) = (1,2,3) + (4,5,6)$ adds no new information – the rank stays at 2 instead of reaching 3.

> **ML connection:** In a dataset, if one feature is a linear combination of others (e.g., `total = price + tax`), the feature matrix has a rank deficiency. This causes the normal equations to be singular. Feature engineering and regularization both address this.

---
### Exercise 5: Rank and Solvability (the Rouche-Capelli Theorem)

We can now state the complete classification of linear systems $Ax = b$ based on rank:

| Condition | Solutions | Geometric meaning |
|-----------|-----------|-------------------|
| $\text{rank}(A) = \text{rank}([A \mid b]) = n$ | **Unique** | $b$ is in the column space, and the columns span all of $\mathbb{R}^n$ |
| $\text{rank}(A) = \text{rank}([A \mid b]) < n$ | **Infinitely many** | $b$ is in the column space, but there are free variables (null space is non-trivial) |
| $\text{rank}(A) < \text{rank}([A \mid b])$ | **No solution** | $b$ is outside the column space – it points in a direction $A$ cannot produce |

Here $[A \mid b]$ is the **augmented matrix** formed by appending $b$ as an extra column.

**Intuition for the rank test:** The augmented matrix has higher rank than $A$ alone precisely when $b$ introduces a new direction not present in the columns of $A$ – meaning $b$ is unreachable.

Implement `classify_system(A, b)` and test all three cases.

In [ ]:
def classify_system(A, b, tol=1e-10):
    """
    Classify the linear system Ax = b.

    Returns
    -------
    str : 'unique', 'infinite', or 'no solution'
    """
    rank_A = np.linalg.matrix_rank(A, tol=tol)
    aug    = np.column_stack([A, b.reshape(-1, 1)])   # [A | b]
    rank_aug = np.linalg.matrix_rank(aug, tol=tol)
    n_vars = A.shape[1]

    if rank_A < rank_aug:
        # b introduces a new direction -- it is outside the column space
        return 'no solution'
    elif rank_A == n_vars:
        # Full column rank and consistent -- columns span enough of space
        return 'unique'
    else:
        # Consistent but rank-deficient -- free variables exist
        return 'infinite'


# --- Test cases ---
# Case 1: unique solution (full rank, square)
A1 = np.array([[2.0, 1.0],
               [1.0, 3.0]])
b1 = np.array([5.0, 10.0])

# Case 2: infinitely many solutions (rank-deficient, b in column space)
A2 = np.array([[1.0, 2.0, 3.0],
               [2.0, 4.0, 6.0]])   # row 2 = 2*row 1 --> rank = 1
b2 = np.array([1.0, 2.0])          # consistent (b2 = 2*b2[0]) so b in col space

# Case 3: no solution (rank-deficient, b NOT in column space)
A3 = np.array([[1.0, 2.0],
               [2.0, 4.0]])        # singular: row 2 = 2*row 1
b3 = np.array([1.0, 3.0])          # inconsistent: 3 != 2*1

cases = [(A1, b1, 'unique'), (A2, b2, 'infinite'), (A3, b3, 'no solution')]
all_passed = True
for A, b, expected in cases:
    result = classify_system(A, b)
    status = 'PASS' if result == expected else 'FAIL'
    print(f'[{status}] Expected "{expected}", got "{result}"')
    all_passed = all_passed and (result == expected)

print()
print('All tests passed.' if all_passed else 'Some tests failed.')

### What we observed

The rank-based classifier correctly identifies all three regimes:

1. **Unique** ($A_1$): A well-conditioned $2 \times 2$ system. Both columns point in independent directions, so every $b$ maps to exactly one $x$.

2. **Infinite** ($A_2$): A $2 \times 3$ system with rank 1. The single constraint $x_1 + 2x_2 + 3x_3 = 1$ defines a 2D plane of solutions in $\mathbb{R}^3$.

3. **No solution** ($A_3$): The two equations $x_1 + 2x_2 = 1$ and $2x_1 + 4x_2 = 3$ are contradictory (the second should give $2 \cdot 1 = 2$, not $3$). The target $b$ points outside the column space.

---
### Overdetermined and underdetermined systems

In ML, we rarely have a perfectly square, full-rank system. Instead we face two common scenarios:

#### Overdetermined: more equations than unknowns ($m > n$)

With more equations than unknowns, the system is typically **inconsistent** – there is no exact solution. Instead, we find the $x$ that minimises the **residual**:

$$x_{\text{ls}} = \arg\min_x \|Ax - b\|^2$$

This is the **least-squares** solution, given by the **normal equations**:

$$x_{\text{ls}} = (A^\top A)^{-1} A^\top b$$

**This is exactly linear regression!** When we fit $y = X\beta + \epsilon$ and minimise $\|X\beta - y\|^2$, we are solving an overdetermined system by least squares.

#### Underdetermined: more unknowns than equations ($m < n$)

With fewer equations than unknowns, there are **infinitely many** exact solutions. Among all solutions, the **minimum-norm** solution has the smallest $\|x\|$:

$$x_{\text{mn}} = A^\top (A A^\top)^{-1} b$$

This uses the **right pseudoinverse** $A^\dagger = A^\top (A A^\top)^{-1}$.

**ML Connection:** Neural networks are massively underdetermined – a typical model has millions of parameters but is trained on fewer effective constraints. Among the infinitely many parameter settings that perfectly fit the training data, the optimisation algorithm (SGD) implicitly selects one. Understanding *which* solution SGD finds is one of the deepest open questions in deep learning theory.

### Exercise 6: Least Squares and Minimum Norm

Implement both solutions manually (using the normal equations) and verify against `np.linalg.lstsq`.

In [ ]:
np.random.seed(42)

# -------------------------------------------------------
# (a) Overdetermined: 8 equations, 3 unknowns
#     This is like linear regression with 8 data points and 3 features.
# -------------------------------------------------------
m_over, n_over = 8, 3
A_over = np.random.randn(m_over, n_over)
x_true = np.array([1.0, 2.0, 3.0])
b_over = A_over @ x_true + 0.1 * np.random.randn(m_over)  # noisy measurements

def least_squares_normal_eq(A, b):
    """
    Solve the least-squares problem min ||Ax - b||^2
    using the normal equations: x = (A^T A)^{-1} A^T b.

    Note: We use np.linalg.solve instead of explicit inverse for stability.
    """
    # A^T A is n x n (small), A^T b is n x 1
    return np.linalg.solve(A.T @ A, A.T @ b)

x_ls_manual = least_squares_normal_eq(A_over, b_over)
x_ls_numpy, _, _, _ = np.linalg.lstsq(A_over, b_over, rcond=None)

print('=== (a) Overdetermined least squares ===')
print(f'  manual (normal eq) : {x_ls_manual}')
print(f'  np.linalg.lstsq    : {x_ls_numpy}')
print(f'  max diff           : {np.max(np.abs(x_ls_manual - x_ls_numpy)):.2e}')
print(f'  residual (manual)  : {np.linalg.norm(A_over @ x_ls_manual - b_over):.6f}')
print()

# -------------------------------------------------------
# (b) Underdetermined: 2 equations, 5 unknowns
#     Infinitely many solutions exist; we want the shortest one.
# -------------------------------------------------------
m_under, n_under = 2, 5
A_under = np.random.randn(m_under, n_under)
x_particular = np.zeros(n_under)
x_particular[:m_under] = 1.0
b_under = A_under @ x_particular  # exact (no noise)

def minimum_norm_solution(A, b):
    """
    Find the minimum-norm solution to Ax = b (underdetermined).
    x = A^T (A A^T)^{-1} b   (right pseudoinverse)

    Among all solutions, this one has the smallest ||x||.
    """
    # A A^T is m x m (small), solve for (A A^T)^{-1} b first
    return A.T @ np.linalg.solve(A @ A.T, b)

x_mn_manual = minimum_norm_solution(A_under, b_under)
x_mn_numpy, _, _, _ = np.linalg.lstsq(A_under, b_under, rcond=None)

print('=== (b) Underdetermined minimum-norm solution ===')
print(f'  manual             : {x_mn_manual}')
print(f'  np.linalg.lstsq    : {x_mn_numpy}')
print(f'  max diff           : {np.max(np.abs(x_mn_manual - x_mn_numpy)):.2e}')
print(f'  ||x_mn|| (manual)  : {np.linalg.norm(x_mn_manual):.6f}')
print(f'  ||x_mn|| (numpy)   : {np.linalg.norm(x_mn_numpy):.6f}')
print()

# Verify x_mn really IS the shortest solution by comparing with a random one
x_rand_particular = x_mn_manual + np.random.randn(n_under) * 0.5
# Project back onto the solution set: ensure A @ x_rand_soln = b
x_rand_soln = (x_rand_particular
               - A_under.T @ np.linalg.inv(A_under @ A_under.T)
               @ (A_under @ x_rand_particular - b_under))
print(f'  ||random soln||    : {np.linalg.norm(x_rand_soln):.6f}  (should be >= ||x_mn||)')
print('Minimum-norm solution verified.' if np.linalg.norm(x_mn_manual) <= np.linalg.norm(x_rand_soln) + 1e-9
      else 'Verification failed.')

### What we observed

**(a) Overdetermined (least squares):**
- Our manual normal equations match `np.linalg.lstsq` to machine precision.
- The residual is non-zero ($\approx 0.18$) because the system is inconsistent (8 noisy equations, 3 unknowns). The least-squares solution minimises this residual.
- The recovered coefficients $[0.94, 2.04, 2.95]$ are close to the true $[1, 2, 3]$ – the noise prevents exact recovery.

**(b) Underdetermined (minimum norm):**
- Among infinitely many solutions to 2 equations in 5 unknowns, the minimum-norm solution has $\|x\| \approx 1.37$.
- A random solution from the same solution set has $\|x\| \approx 1.62$ – larger, as guaranteed by the theory.
- The minimum-norm solution is the unique point in the solution set closest to the origin.

> **Deep learning insight:** When a neural network has many more parameters than training examples, gradient descent on the training loss converges to a solution that interpolates the data (training loss = 0). The implicit bias of gradient descent tends to select solutions with certain regularity properties – analogous to (but more complex than) minimum norm. This is thought to be one reason deep networks generalise despite massive overparameterisation.

---
## Summary

### Concepts exercised in this lab

| Exercise | Concept |
|----------|---------|
| 1 – 2x2 formula | Determinant, cofactor formula for $A^{-1}$ |
| 2 – Gauss-Jordan | Elementary row operations, RREF |
| 3 – Singular matrices | Determinant = 0, condition number = infinity |
| 4 – Linear independence | Rank test for independence |
| 5 – Rouche-Capelli | Rank conditions for solvability |
| 6 – Least squares / min-norm | Normal equations, right pseudoinverse |
| 7 – Condition number | Perturbation theory, numerical stability |

### Key takeaways from Sections 2.3–2.4

- The inverse $A^{-1}$ exists if and only if $A$ is **square and full-rank** ($\det A \neq 0$).
- In practice **never form** $A^{-1}$ just to solve $Ax = b$; use `np.linalg.solve` instead.
- A system $Ax = b$ is solvable only when $b$ lies in the **column space** (span) of $A$.
- **Linear independence** determines the rank, which determines the dimension of the column space.
- The **condition number** $\kappa(A) = \sigma_{\max}/\sigma_{\min}$ quantifies how much numerical errors are amplified; large $\kappa$ signals an ill-conditioned problem.
- When no exact solution exists (**overdetermined**), the **least-squares** solution minimises the residual.
- When infinitely many solutions exist (**underdetermined**), the **minimum-norm** solution is the shortest.
- **ML connections:** The normal equations are linear regression. Ill-conditioning motivates regularization. Underdetermined systems describe overparameterised neural networks.

### Next

**Lab 03** – Eigenvalues, Eigenvectors, and Matrix Decompositions (Book Sections 2.7–2.8)